## 1. Imports and paths

You may start Jupyter from the project root or the `notebooks/` folder. The cell below supports both.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.mmm_response import ResponseModel, LABELS
from src.budget_optimization import (build_bounds, optimize_budget,
    allocation_comparison, evaluate_scenarios, sensitivity_analysis, run_pipeline)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the MMM team's response functions and historical spends

The final MMM is calibrated using `models/mmm/final_mmm.json`. `current_spend` is the model's recorded **2025 average monthly spending** across six channels.

In [2]:
model = ResponseModel(ROOT / "models" / "mmm" / "final_mmm.json")
history = pd.read_csv(ROOT / "data" / "processed" / "mmm_analysis_ready.csv")
current = model.current_spend
fixed_budget_lkr = sum(current.values())
current_incremental_revenue_lkr = model.total_revenue(current)
print(f"Budget: LKR {fixed_budget_lkr:,.2f}")
print(f"Current MMM incremental revenue: LKR {current_incremental_revenue_lkr:,.2f}")
print(f"Current portfolio ROAS: {current_incremental_revenue_lkr/fixed_budget_lkr:.3f}x")

Budget: LKR 20,023,825.00
Current MMM incremental revenue: LKR 48,026,843.75
Current portfolio ROAS: 2.398x


## 3. Specify defensible minimum/maximum channel spend limits

**Assumption to review with the team:** each channel may move by up to 30% relative to its 2025 average, but not outside its recorded historical monthly spend range. The limits prevent unsupported extrapolation and prohibit cutting Radio to zero just because its fitted coefficient is zero.

In [3]:
bounds = build_bounds(model, history, lower_ratio=0.70, upper_ratio=1.30, cap_to_history=True)
bound_table = pd.DataFrame([
    {"Channel": LABELS[c], "Min LKR M": bounds[c][0]/1e6,
     "Current LKR M": current[c]/1e6, "Max LKR M": bounds[c][1]/1e6}
    for c in model.channels
])
display(bound_table)

,Channel,Min LKR M,Current LKR M,Max LKR M
0,TV,3.30,4.71,6.02
1,Facebook/Instagram,3.18,4.54,5.90
2,Google Search,2.52,3.60,4.68
3,YouTube,1.79,2.55,3.18
4,Radio,1.28,1.83,2.17
5,Outdoor,1.95,2.78,3.62


## 4. Define the optimisation problem

\[\max_{x_1,\ldots,x_6}\sum_{i=1}^{6} R_i(x_i)\]

Subject to \(\sum_i x_i = B\) and \(L_i \le x_i \le U_i\).

The response \(R_i\) is Team 2's fitted Hill saturation model and uses *steady-state* normalised adstock. We solve it with SciPy SLSQP and its analytic gradient. With the final model's Hill alpha=1 and nonnegative media coefficients, it is a concave revenue objective.

In [4]:
result = optimize_budget(model, bounds, fixed_budget_lkr)
comparison = allocation_comparison(model, result)
assert np.isclose(sum(result.allocation.values()), fixed_budget_lkr, atol=0.1)
assert result.incremental_revenue_lkr >= current_incremental_revenue_lkr - 0.1
print(result.message)
print(f"Optimized predicted incremental revenue: LKR {result.incremental_revenue_lkr:,.2f}")
print(f"Predicted uplift: LKR {result.incremental_revenue_lkr-current_incremental_revenue_lkr:,.2f} "
      f"({100*(result.incremental_revenue_lkr/current_incremental_revenue_lkr-1):.2f}%)")
print(f"Optimized portfolio ROAS: {result.portfolio_roas:.3f}x")
display(comparison[["channel", "current_spend_lkr", "optimized_spend_lkr", "spend_change_pct",
                    "current_marginal_roas", "optimized_marginal_roas"]])

Optimization terminated successfully
Optimized predicted incremental revenue: LKR 50,142,584.33
Predicted uplift: LKR 2,115,740.58 (4.41%)
Optimized portfolio ROAS: 2.504x


,channel,current_spend_lkr,optimized_spend_lkr,spend_change_pct,current_marginal_roas,optimized_marginal_roas
0,TV,"4,711,485.17","4,084,058.40",-13.32,0.48,0.55
1,Facebook/Instagram,"4,540,710.92","3,178,497.64",-30.00,0.41,0.54
2,Google Search,"3,601,297.58","4,681,686.86",30.00,1.27,0.88
3,YouTube,"2,553,401.83","3,178,264.00",24.47,1.16,0.84
4,Radio,"1,834,483.75","1,284,138.62",-30.00,0.00,0.00
5,Outdoor,"2,782,445.75","3,617,179.48",30.00,1.91,1.32


## 5. Chart: current vs optimal spend

In [5]:
comparison.set_index("channel")[["current_spend_lkr", "optimized_spend_lkr"]].div(1e6).plot(
    kind="bar", figsize=(11, 5), rot=15,
    title="Current vs optimized monthly spend (LKR million)")
plt.ylabel("Monthly spend (LKR million)")
plt.tight_layout()
plt.show()

C:\Users\thara\AppData\Local\Temp\ipykernel_12028\2936443319.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Budget scenarios (±10%, ±20%)

Run the same optimizer with a changed total budget. **These changes mix budget-level effects with reallocation effects**; the fixed-budget +4.41% result is the clean reallocation comparison.

In [6]:
scenarios, scenario_allocations = evaluate_scenarios(model, bounds)
display(scenarios)
scenarios.plot(x="budget_lkr", y="predicted_incremental_revenue_lkr", marker="o",
               figsize=(8, 4), legend=False, title="Optimized revenue vs available budget")
plt.ylabel("Modelled monthly incremental revenue (LKR)")
plt.xlabel("Total monthly advertising budget (LKR)")
plt.grid(alpha=0.2)
plt.show()

,budget_multiplier,budget_change_pct,budget_lkr,predicted_incremental_revenue_lkr,predicted_portfolio_roas,revenue_change_vs_current_lkr,revenue_change_vs_current_pct
0,0.80,-20.00,"16,019,060.00","45,788,433.53",2.86,"-2,238,410.22",-4.66
1,0.90,-10.00,"18,021,442.50","48,481,587.54",2.69,"454,743.79",0.95
2,1.00,0.00,"20,023,825.00","50,142,584.33",2.50,"2,115,740.58",4.41
3,1.10,10.00,"22,026,207.50","51,129,540.81",2.32,"3,102,697.05",6.46
4,1.20,20.00,"24,028,590.00","51,938,785.98",2.16,"3,911,942.23",8.15


C:\Users\thara\AppData\Local\Temp\ipykernel_12028\3148808945.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Sensitivity checks

Do recommendations still look useful when channel spend limits change or when the model is fitted only on the training period? This is an exploratory robustness test—not a confidence interval.

In [7]:
constraint_sensitivity, model_sensitivity = sensitivity_analysis(model, history)
display(constraint_sensitivity)
display(model_sensitivity)

,constraint_scenario,lower_ratio,upper_ratio,budget_lkr,predicted_incremental_revenue_lkr,predicted_revenue_lift_pct
0,conservative,0.85,1.15,"20,023,825.00","49,281,116.20",2.61
1,moderate,0.70,1.30,"20,023,825.00","50,142,584.33",4.41
2,flexible,0.60,1.50,"20,023,825.00","50,340,733.40",4.82


,model_variant,current_revenue_lkr,revenue_using_full_model_allocation_lkr,lift_full_model_allocation_pct,revenue_using_own_optimum_lkr,lift_own_optimum_pct
0,full_data_refit,"48,026,843.75","50,142,584.33",4.41,"50,142,584.33",4.41
1,train_only_fit,"50,064,437.18","52,508,867.90",4.88,"52,508,867.90",4.88


## 8. Export results for Team 3, the dashboard, and final report

Exports include tidy CSVs and presentation figures. These are generated from the same code as the standalone CLI.

In [ ]:
final_result, summary = run_pipeline(
    output_dir=ROOT / "outputs" / "budget_optimization",
    figures_dir=ROOT / "reports" / "figures")
display(summary)
print("Saved CSV tables under outputs/budget_optimization/ and charts under reports/figures/11_*.png")

,budget_lkr,current_predicted_incremental_revenue_lkr,optimized_predicted_incremental_revenue_lkr,predicted_revenue_lift_lkr,predicted_revenue_lift_pct,current_portfolio_roas,optimized_portfolio_roas
0,"20,023,825.00","48,026,843.75","50,142,584.33","2,115,740.58",4.41,2.40,2.50


Saved CSV tables under outputs/budget_optimization/ and charts under reports/figures/10_*.png


## 9. Interpretation and limits

- Maximize *MMM-estimated incremental revenue*, not profit. Profit ROI requires valid gross/contribution margins and other costs.
- This scenario represents monthly *steady-state* spending (normalised adstock), not the first month following changes.
- Radio's coefficient is zero in this fit, which should not be interpreted as evidence that Radio truly has no effect.
- Monthly MMM data include only 36 rows, with potential media collinearity and attribution uncertainty. Treat results as a prioritised **pilot hypothesis**, not a guaranteed improvement.
- Validate feasibility of minimum/maximum channel spends with the business and monitor actual experimental outcomes before rollout.